# ArchAI SLM Training: Phi-3.5 3.8B Fine-Tuning
This notebook is optimized for **Google Colab (Free T4)** and **Kaggle (Dual T4)**. It uses **Unsloth** to perform 4-bit QLoRA fine-tuning on the synthetic Enterprise Architecture corpus.

In [1]:
# Mount Drive (optional but recommended for saving model)
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes datasets huggingface_hub

# Clone your repo (if not already)
!git clone https://github.com/MaheshUmale/ArchAI---Enterprise-AI-Solution-Architect.git
%cd ArchAI---Enterprise-AI-Solution-Architect
%%capture
# 1. Install Unsloth and dependencies
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 23.9 MB/s eta 0:00:00
Cloning into 'ArchAI---Enterprise-AI-Sol

UsageError: Line magic function `%%capture` not found.


In [2]:
# Mount Drive (optional but recommended for saving model)
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes datasets huggingface_hub

# Clone your repo (if not already)
!git clone https://github.com/MaheshUmale/ArchAI---Enterprise-AI-Solution-Architect.git
%cd ArchAI---Enterprise-AI-Solution-Architect

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into 'ArchAI---Enterprise-AI-Solution-Architect'...
remote: Enumerating objects: 652, done.
remote: Counting objects: 100% (306/306), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 652 (delta 160), reused 196 (delta 83), pack-reused 346 (from 1)
Receiving objects: 100% (652/652), 161.36 MiB | 19.50 MiB/s, done.
Resolving deltas: 100% (237/237), done.
/content/ArchAI---Enterprise-AI-Solution-Architect/ArchAI---Enterprise-AI-Solution-Architect


In [3]:
# Install script requirements
!pip install -q -r scripts/requirements_slm.txt

# 1. Ingest sources
!python scripts/ingest_master_sources.py --doc_dir docs --output backend/data/raw_knowledge.jsonl

# 2. Generate synthetic EA corpus (800-1200 samples for NANO)
!python scripts/generate_ea_corpus.py \
  --total_count 1000 \
  --model groq/llama-3.3-70b \
  --output backend/data/synthetic_corpus.jsonl \
  --skills "TOGAF,Zachman,AWS_WAF,C4_Model,GoF_EIP,Microservices"

# 3. Clean & filter
!python scripts/filter_diversity.py --input backend/data/synthetic_corpus.jsonl --output backend/data/optimized.json
!python scripts/clean_boilerplate.py --input backend/data/optimized.json --output backend/data/cleaned_corpus.json
!python scripts/precheck_tokenization.py --input backend/data/cleaned_corpus.json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.5 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/content/ArchAI---Enterprise-AI-Solution-Architect/ArchAI---Enterprise-AI-Solution-Architect/scripts/ingest_master_sources.py", line 8, in <module>
    import pdfplumber
ModuleNotFoundError: No module named 'pdfplumber'
2026-05-21 16:55:19,134 - ERROR - Failed to import backend dependencies: No module named 'langchain_openai'
2026-05-21 16:55:27,147 - INFO - NumExpr defaulting to 2 threads.
2026-05-21 16:55:34,937 - WARNING - Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-05-21 16:55:37,500 - INFO - TensorFlow version 2.20.0 available.
2026-05-21 16:55:37,501 - INFO - JAX version 0.7.2 available.
2026-05-21 16:55:42,423 - INFO - Loading local embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-05-21 16:55:42,735 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transf

In [6]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt   # Important!

# ============================
# 1. Load Model (NANO)
# ============================
model_name = "unsloth/Phi-3-mini-4k-instruct"   # Best for Colab T4 + NANO

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# ============================
# 2. Load & Prepare Dataset
# ============================
dataset = load_dataset("json", data_files="backend/data/cleaned_corpus.json", split="train")

# Convert ShareGPT format if needed
# dataset = standardize_sharegpt(dataset)

# Formatting function for conversations
def formatting_func(examples):
    convos = examples["conversations"]          # or "messages" — check your JSON structure
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
             for convo in convos]
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_func, batched=True)

# ============================
# 3. Trainer (Fixed)
# ============================
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",        # Now exists after formatting
    max_seq_length = 4096,
    dataset_num_proc = 2,
    packing = True,                     # Important for efficiency
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 150,                # Adjust based on your corpus size (~1-2 hrs on T4)
        learning_rate = 3e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",             # Disable wandb if not set up
    ),
)

trainer.train()

==((====))==  Unsloth 2026.5.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Map:   0%|          | 0/247 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/247 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 247 | Num Epochs = 10 | Total steps = 150
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 14,942,208 of 3,836,021,760 (0.39% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-150/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-150.


TrainOutput(global_step=150, training_loss=0.0, metrics={'train_runtime': 185.1699, 'train_samples_per_second': 12.961, 'train_steps_per_second': 0.81, 'total_flos': 52003869308928.0, 'train_loss': 0.0, 'epoch': 9.387096774193548})

In [7]:
model.save_pretrained("archai_nano")
tokenizer.save_pretrained("archai_nano")

# GGUF export (best for Ollama / llama.cpp)
model.save_pretrained_gguf("archai_nano_gguf", tokenizer, quantization_method="q4_k_m")

Unsloth: Restored added_tokens_decoder metadata in archai_nano/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in archai_nano.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in archai_nano_gguf/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in archai_nano_gguf.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [01:09<01:09, 69.94s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:47<00:00, 53.53s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:56<00:00, 88.35s/it]


Unsloth: Merge process complete. Saved to `/content/ArchAI---Enterprise-AI-Solution-Architect/ArchAI---Enterprise-AI-Solution-Architect/archai_nano_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['archai_nano_gguf_gguf/phi-3-mini-4k-instruct.F16.gguf']
U

{'save_directory': 'archai_nano_gguf',
 'gguf_directory': 'archai_nano_gguf_gguf',
 'gguf_files': ['archai_nano_gguf_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'archai_nano_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

### Load Optimized Config
We use the settings defined in `configs/phi35-qlora.yml` for memory-efficient training.

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 4096 # Matches configs/phi35-qlora.yml
dtype = None # Auto detection
load_in_4bit = True # Use 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32, # Matches configs/phi35-qlora.yml
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "phi-3",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"},
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Upload your synthetic_corpus_cleaned.json to the Colab environment
dataset = load_dataset("json", data_files="synthetic_corpus_cleaned.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True, # Matches configs/phi35-qlora.yml sample_packing
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine", # Matches configs/phi35-qlora.yml
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Save the fine-tuned adapter
model.save_pretrained("archai_phi35_adapter")
tokenizer.save_pretrained("archai_phi35_adapter")

# Optional: Merge to 16bit and save to GGUF for local inference
# model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")